# ACT Training on comma2k19

This notebook converts an extracted comma2k19 chunk into a local LeRobotDataset and trains an ACT policy with LeRobot's training CLI.

The conversion step is recommended because repeatedly streaming and decoding comma2k19 HEVC videos during training is inefficient.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
CHUNK_PATH = PROJECT_DIR / "comma2k19_data" / "extracted" / "Chunk_1"
DATASET_REPO_ID = "local/comma2k19_act"
DATASET_ROOT = PROJECT_DIR / "lerobot_datasets" / DATASET_REPO_ID
OUTPUT_DIR = PROJECT_DIR / "outputs" / "train" / "act_comma2k19"

print("Project:", PROJECT_DIR)
print("Chunk:", CHUNK_PATH)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## Install / Check Dependencies

LeRobot must be installed in the notebook environment. If this import fails, install LeRobot before running the conversion and training cells.

In [ ]:
try:
    import lerobot
    print("LeRobot import OK", getattr(lerobot, "__version__", "unknown version"))
except Exception as exc:
    raise RuntimeError("Install LeRobot first, for example: pip install lerobot") from exc

## Optional Smoke-Test Conversion

This creates a tiny dataset with two episodes and a short frame cap. Run it first to verify paths and video encoding before converting the full chunk.

In [ ]:
!python convert_comma2k19_to_lerobot.py \
  --chunk-path "{CHUNK_PATH}" \
  --repo-id "local/comma2k19_act_smoke" \
  --output-root "{PROJECT_DIR / 'lerobot_datasets'}" \
  --max-episodes 2 \
  --max-frames-per-episode 64 \
  --overwrite

## Convert Full Dataset

This writes the local LeRobotDataset used by ACT training. Use no history if you want state dimension 2, or use the default history offsets for richer state dimension 10.

In [ ]:
!python convert_comma2k19_to_lerobot.py \
  --chunk-path "{CHUNK_PATH}" \
  --repo-id "{DATASET_REPO_ID}" \
  --output-root "{PROJECT_DIR / 'lerobot_datasets'}" \
  --width 256 \
  --height 256 \
  --fps 20 \
  --future-time 1.0 \
  --overwrite

## Inspect Converted Dataset

In [ ]:
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset


dataset = LeRobotDataset(DATASET_REPO_ID, root=DATASET_ROOT)
print(dataset)
print(dataset.features)
sample = dataset[0]
print("state", sample["observation.state"].shape)
print("action", sample["action"].shape)
print("image", sample["observation.images.front"].shape)

## Train ACT

ACT reads the observation/action dimensions from the saved LeRobotDataset metadata.

In [ ]:
!python train_act_policy.py \
  --dataset-repo-id "{DATASET_REPO_ID}" \
  --dataset-root "{DATASET_ROOT}" \
  --output-dir "{OUTPUT_DIR}" \
  --job-name act_comma2k19 \
  --device cuda \
  --batch-size 8 \
  --steps 10000 \
  --num-workers 4

## Resume Training

After a run starts, LeRobot saves checkpoints under outputs/train/act_comma2k19/checkpoints.

In [ ]:
# Example resume command. Uncomment after a checkpoint exists.
# !lerobot-train \
#   --config_path "{OUTPUT_DIR / 'checkpoints' / 'last' / 'pretrained_model' / 'train_config.json'}" \
#   --resume=true